# Staged CO₂ surrogate comparison

One Save & Run All workflow: pressure-prior diagnostic, learned pressure hybrids, constrained training, three-stage pushforward, ConvLSTM controls, and CCSNet–GNN dual branch. The saved dataset already includes synthetic regimes and SPE10 Tarbert/Upper Ness x–z cases. Results are against the exploratory 2D solver, not field accuracy.

In [ ]:
from pathlib import Path
import subprocess, sys, os, json, shutil, pandas as pd, torch
REPO = Path('/kaggle/working/surrogate_repo')
BRANCH = 'codex/ccs-lstm-fno-variants'
if not REPO.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch','https://github.com/smerarawal/shale-well-game-theory-codes.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO),'checkout',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin',BRANCH], check=True)
SEQ = REPO / 'sequential_game'
print('Code commit:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','--short','HEAD'], text=True).strip())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none; switch Kaggle accelerator to GPU T4')

In [ ]:
INPUT = Path('/kaggle/input')
data_candidates = [p.parent.parent for p in INPUT.rglob('friendly.npz') if p.parent.name == 'train' and (p.parent.parent/'validation'/'friendly.npz').exists()]
if not data_candidates: raise FileNotFoundError('Attach the saved co2_xz_dataset_v2 notebook output as an input.')
data_candidates.sort(key=lambda p: ('co2_xz_dataset_v2' not in str(p).lower(), len(str(p))))
DATA_DIR = data_candidates[0]
required_ckpts = [f'co2_{b}_s{s}.pt' for b in ('gnn','ccs') for s in (0,1,2)]
base_roots = sorted({p.parent for p in INPUT.rglob('co2_gnn_s0.pt')})
BASE_DIR = next((root for root in base_roots if all((root/name).exists() for name in required_ckpts)), None)
if BASE_DIR is None: raise FileNotFoundError('Attach the prior CO2 surrogate results output containing GNN and CCS checkpoints for seeds 0, 1, 2.')
OUT = Path('/kaggle/working/co2_staged_results')
OUT.mkdir(parents=True, exist_ok=True)
print('Data:', DATA_DIR)
print('Baseline checkpoints:', BASE_DIR)
print('Outputs:', OUT)

## Stage 0: select a fixed prior substep count using validation only
The path diagnostic below uses labelled endpoint saturation as an oracle to estimate an upper bound; it is not a deployable prior2 score. The chosen count is selected from the frozen-prior validation pressure error, never from the test split.

In [ ]:
DIAG = OUT / 'prior_diagnostic'
subprocess.run([sys.executable,str(SEQ/'diagnose_co2_pressure_prior.py'),'--data-dir',str(DATA_DIR),'--out-dir',str(DIAG),'--n-sub','4','8','16','--samples-per-cohort','24'], check=True)
d = pd.read_csv(DIAG/'pressure_prior_diagnostic.csv')
frozen = d[d.mode == 'frozen'].groupby('n_sub').pmax_abs_mean_pa.mean()
N_SUB = int(frozen.idxmin())
print('Validation-selected fixed substeps:', N_SUB)
display(d)

## Stage 1: frozen, moving-path, and prior-lite pressure hybrids
The correction network learns the residual pressure field. Other state fields remain from the matched frozen baseline so this comparison isolates pressure-prior effects. `path` uses the base model’s predicted endpoint saturation, not target saturation.

In [ ]:
PRIOR_OUT = OUT/'pressure_hybrids'
subprocess.run([sys.executable,str(SEQ/'train_co2_pressure_hybrid.py'),'--data-dir',str(DATA_DIR),'--base-dir',str(BASE_DIR),'--out-dir',str(PRIOR_OUT),'--backbones','gnn','ccs','--modes','lite','frozen','path','--seeds','0','1','2','--n-sub',str(N_SUB),'--epochs','40','--device','cuda' if torch.cuda.is_available() else 'cpu'], check=True)

## Stage 2: constrained controls and three-stage pushforward
Both one-step and unroll-3 models use the same soft mass and validity penalties. The pushforward window detaches predicted states between stages to keep T4 memory bounded. The baseline trainer still selects checkpoints on validation one-step loss; compare six-stage test rollout metrics before choosing a winner.

In [ ]:
COMMON = ['--data-dir',str(DATA_DIR),'--seeds','0','1','2','--epochs','60','--device','cuda' if torch.cuda.is_available() else 'cpu','--mass-loss-weight','0.05','--valid-loss-weight','1.0']
CTRL_OUT = OUT/'constrained_one_step'
subprocess.run([sys.executable,str(SEQ/'train_co2_xz_surrogate.py'),*COMMON,'--out-dir',str(CTRL_OUT),'--backbones','gnn','ccs','--rollout-unroll','1'], check=True)
UNROLL_OUT = OUT/'constrained_unroll3'
subprocess.run([sys.executable,str(SEQ/'train_co2_xz_surrogate.py'),*COMMON,'--out-dir',str(UNROLL_OUT),'--backbones','gnn','ccs','--rollout-unroll','3'], check=True)

## Stage 3: temporal and dual-branch candidates
The ConvLSTM variants are trained on the same three-stage windows as the non-recurrent controls. This tests whether hidden state helps beyond explicit Markov state and pushforward training. The dual branch is also trained with the same physical penalties.

In [ ]:
TEMP_OUT = OUT/'constrained_convlstm_unroll3'
subprocess.run([sys.executable,str(SEQ/'train_co2_xz_surrogate.py'),*COMMON,'--out-dir',str(TEMP_OUT),'--backbones','ccs_lstm','gnn_lstm','--rollout-unroll','3'], check=True)
DUAL_OUT = OUT/'constrained_dual_branch'
subprocess.run([sys.executable,str(SEQ/'train_co2_xz_surrogate.py'),*COMMON,'--out-dir',str(DUAL_OUT),'--backbones','ccs_gnn','--rollout-unroll','3'], check=True)

In [ ]:
records=[]
for path in OUT.rglob('*_metrics.json'):
    try: records.append(json.loads(path.read_text()))
    except Exception as exc: print('Skipping incomplete metric file', path, exc)
(OUT/'all_stage_metrics.json').write_text(json.dumps(records, indent=2))
rows=[]
for rec in records:
    for split_key in ('one_step_test','autoregressive_test'):
        for geology, metrics in rec.get(split_key,{}).items():
            if not isinstance(metrics,dict): continue
            common={k:rec.get(k) for k in ('backbone','mode','seed','n_sub','best_epoch')}
            row={**common,'evaluation':split_key,'geology':geology,'stage':'all'}
            row.update({k:v for k,v in metrics.items() if not isinstance(v,(dict,list))})
            rows.append(row)
            for stage, sm in metrics.get('by_stage',{}).items():
                sr={**common,'evaluation':split_key,'geology':geology,'stage':stage}
                sr.update({k:v for k,v in sm.items() if not isinstance(v,(dict,list))})
                rows.append(sr)
table=pd.DataFrame(rows)
table.to_csv(OUT/'metrics_long.csv',index=False)
print('Metric records:', len(records), '| table rows:', len(table))
display(table.head(30))
print('Saved results under:', OUT)
display(pd.DataFrame([{'file':p.name,'bytes':p.stat().st_size} for p in OUT.rglob('*') if p.is_file()]).tail(40))